# 演習6. 容量とバックプレッシャ ―― キューをいくつにするか

## シーン

演習5で、容量つきのキューが完成しました。使うときには、必ずこう書きます。

```cpp
BoundedQueue<Frame> q(?);      // ← ここに何を入れるか
```

**この数字を、何を根拠に決めればよいのでしょうか。**

「大きいほうが安全そう」「小さいほうが速そう」――どちらも、たぶん間違いです。
この演習では、容量を変えると**何が変わって、何が変わらないのか**を測ります。

見る尺度は3つです。

- **スループット** … 全部処理し終わるまでの時間。短いほどよい
- **遅れ（レイテンシ）** … 1個のデータがキューに居た時間。短いほどよい
- **メモリ** … キューに並んだ最大の個数。少ないほどよい

この3つが、容量に対してまったく違うふるまいをします。そこが面白いところです。

## 6-1. 【予測クイズ】容量を変えると、速くなるか

作る側が **10ms に1個**、受け取る側が **1個 50ms**。作る側のほうが5倍速い設定です。
容量を 1 / 4 / 32 / 1000 と変えて、60個流します。

**実行する前に予測してください。**

- **所要時間** は、容量を大きくするとどうなるでしょうか
- **平均の遅れ** は、容量を大きくするとどうなるでしょうか

In [ ]:
%%writefile ex06a.cpp
#include <iostream>
#include <iomanip>
#include <thread>
#include <queue>
#include <mutex>
#include <condition_variable>
#include <chrono>
using namespace std::chrono;

// 演習5で完成させたキュー（観察用に「最大の長さ」も数える）
template <typename T>
class BoundedQueue {
public:
    explicit BoundedQueue(std::size_t capacity) : capacity_(capacity) {}
    void push(const T& v) {
        std::unique_lock<std::mutex> lk(mtx_);
        can_push_.wait(lk, [this] { return q_.size() < capacity_; });
        q_.push(v);
        if (q_.size() > peak_) peak_ = q_.size();
        lk.unlock(); can_pop_.notify_one();
    }
    T pop() {
        std::unique_lock<std::mutex> lk(mtx_);
        can_pop_.wait(lk, [this] { return !q_.empty(); });
        T v = q_.front(); q_.pop();
        lk.unlock(); can_push_.notify_one();
        return v;
    }
    std::size_t peak() const { return peak_; }
private:
    std::queue<T> q_;
    std::size_t capacity_, peak_ = 0;
    mutable std::mutex mtx_;
    std::condition_variable can_pop_, can_push_;
};

const int N = 60;                       // 流すデータの個数
void wait_ms(int ms) { std::this_thread::sleep_for(milliseconds(ms)); }

void run(std::size_t capacity) {
    BoundedQueue<steady_clock::time_point> q(capacity);
    long long stay_us = 0;                                   // キューに居た時間の合計

    auto t0 = steady_clock::now();
    std::thread producer([&] {
        for (int i = 0; i < N; i++) { wait_ms(10); q.push(steady_clock::now()); }
    });
    std::thread consumer([&] {
        for (int i = 0; i < N; i++) {
            auto born = q.pop();
            stay_us += duration_cast<microseconds>(steady_clock::now() - born).count();
            wait_ms(50);
        }
    });
    producer.join(); consumer.join();
    int ms = duration_cast<milliseconds>(steady_clock::now() - t0).count();

    std::cout << std::right
              << std::setw(8) << capacity
              << std::setw(12) << ms
              << std::setw(12) << q.peak()
              << std::setw(12) << std::fixed << std::setprecision(0) << (stay_us / N / 1000.0)
              << "\n";
}

int main() {
    std::cout << "作る側 10ms に1個、受け取る側 1個 50ms、" << N << "個流す\n\n";
    std::cout << "  capacity   time(ms)    peak(個)   delay(ms)\n";
    std::cout << "-------------------------------------------------\n";
    run(1);
    run(4);
    run(32);
    run(1000);        // 実質、上限なし

    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ex06a.cpp -o ex06a && ./ex06a

### 結果 ―― 速さは1ミリも変わらない。遅れだけが増える

だいたい次のようになったはずです。

```
  capacity   time(ms)    peak(個)   delay(ms)
-------------------------------------------------
       1        3024           1          88
       4        3021           4         227
      32        3020          32        1067
    1000        3023          48        1183
```

- **所要時間 … どれも約 3020ms。まったく変わりません**
- **最大の行列 … 容量そのまま。容量が上限として効いています**
- **平均の遅れ … 88ms → 1183ms。13倍以上に悪化しています**

所要時間が変わらない理由は、演習1で見たとおりです。
全体の速さを決めているのは**一番遅い段**（ここでは 50ms の受け取る側）で、
60個 × 50ms = 3000ms。作る側がどれだけ先へ進もうと、これは動きません。

**先へ進んだ分は、速さにならずにキューへ積み上がっただけ**です。

### 容量を付けるとは、何をすることか

満杯のとき、入れる側は `can_push_.wait(...)` で待たされます。
つまり**遅い側の都合が、速い側に伝わって、速い側を減速させます。**
これを **バックプレッシャ（backpressure、逆向きの圧力）** と呼びます。

```
【容量なし】速い側は止まらない。行列だけが伸びる
作る側   ■■■■■■■■■■■■■■■■■■■■■■■■
行列     ▁▂▃▄▅▆▇███████████████
受ける側 ■■■■■■■■■■■■■■■■■■■■■■■■

【容量4】満杯になると、速い側が待たされる
作る側   ■■■■...■...■...■...■...■...
行列     ▁▂▃▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
受ける側 ■■■■■■■■■■■■■■■■■■■■■■■■
```

下の行（受ける側）は、どちらもびっしり埋まっています。**仕事量は同じ**です。
違うのは、**待たされているのが誰か**だけです。

### なぜ「遅れ」が大事なのか

所要時間が同じなら、遅れが増えても構わないように思えるかもしれません。
**リアルタイム処理では致命的です。**

キューに30個並んでいて、1個の処理に50msかかるなら、
いまカメラが撮った映像が画面に出るのは **1.5秒後** です。
録画の再生なら問題ありませんが、その場で見るものとしては使いものになりません。

### メモリの話

さらに、容量を決めないと**メモリが尽きます**。

- 1フレームが 512×256×3バイト ≒ **384KB**
- 作る側が受け取る側より毎秒15フレーム速いなら、毎秒 5.8MB ずつ増える
- 1分で 340MB、10分で 3.4GB

「動くけれど、しばらく走らせると落ちる」という、いちばん見つけにくい壊れ方をします。

> **容量を決めないということは、遅れとメモリの上限を決めないということ。**

## 6-2. 【予測クイズ】では、容量は 1 が最善か

6-1 の結果だけを見ると、「遅れが最小なので容量1がいちばん良い」と言えそうです。
**本当でしょうか。**

いままでの設定は、作る側も受け取る側も**毎回きっちり同じ時間**でした。
現実はそうなりません。フレームによって写っているものが違えば、推論時間も変わります。

そこで、作る側を**でこぼこ**にします。

- 作る側 … ふだんは 5ms で1個。ただし **5個に1回だけ 105ms** かかる（平均 25ms/個）
- 受け取る側 … いつも 30ms/個

**平均で見れば、作る側のほうが速い**（25ms < 30ms）ことに注意してください。
ですから理想の所要時間は、受け取る側で決まって 60 × 30 = **1800ms** のはずです。

`idle` は「受け取る側が、次のデータを待って手待ちしていた時間の合計」です。

**実行する前に予測してください。** 容量1のとき、所要時間は 1800ms になるでしょうか。

In [ ]:
%%writefile ex06b.cpp
#include <iostream>
#include <iomanip>
#include <thread>
#include <queue>
#include <mutex>
#include <condition_variable>
#include <chrono>
using namespace std::chrono;

template <typename T>
class BoundedQueue {
public:
    explicit BoundedQueue(std::size_t capacity) : capacity_(capacity) {}
    void push(const T& v) {
        std::unique_lock<std::mutex> lk(mtx_);
        can_push_.wait(lk, [this] { return q_.size() < capacity_; });
        q_.push(v);
        lk.unlock(); can_pop_.notify_one();
    }
    T pop() {
        std::unique_lock<std::mutex> lk(mtx_);
        can_pop_.wait(lk, [this] { return !q_.empty(); });
        T v = q_.front(); q_.pop();
        lk.unlock(); can_push_.notify_one();
        return v;
    }
private:
    std::queue<T> q_;
    std::size_t capacity_;
    mutable std::mutex mtx_;
    std::condition_variable can_pop_, can_push_;
};

const int N = 60;
void wait_ms(int ms) { std::this_thread::sleep_for(milliseconds(ms)); }

// 作る側：ふだんは 5ms で1個。ただし 5個に1回だけ 105ms かかる
//         平均は 25ms/個 なので、受け取る側(30ms)より「平均では速い」
int make_time(int i) { return (i % 5 == 4) ? 105 : 5; }

void run(std::size_t capacity) {
    BoundedQueue<steady_clock::time_point> q(capacity);
    long long idle_us = 0;                          // 受け取る側が手待ちだった時間
    long long stay_us = 0;                          // データがキューに居た時間

    auto t0 = steady_clock::now();
    std::thread producer([&] {
        for (int i = 0; i < N; i++) { wait_ms(make_time(i)); q.push(steady_clock::now()); }
    });
    std::thread consumer([&] {
        for (int i = 0; i < N; i++) {
            auto w0 = steady_clock::now();
            auto born = q.pop();                     // ここで待たされた時間が「手待ち」
            auto now = steady_clock::now();
            idle_us += duration_cast<microseconds>(now - w0).count();
            stay_us += duration_cast<microseconds>(now - born).count();
            wait_ms(30);
        }
    });
    producer.join(); consumer.join();
    int ms = duration_cast<milliseconds>(steady_clock::now() - t0).count();

    std::cout << std::right << std::setw(8) << capacity
              << std::setw(12) << ms
              << std::setw(12) << (idle_us / 1000)
              << std::setw(13) << (stay_us / N / 1000) << "\n";
}

int main() {
    std::cout << "作る側 : ふだん 5ms、5個に1回だけ 105ms（平均 25ms/個）\n";
    std::cout << "受け取る側 : いつも 30ms/個\n";
    std::cout << "受け取る側がボトルネックなので、理想は " << N << " x 30 = " << N * 30 << " ms\n\n";
    std::cout << "  capacity   time(ms)   idle(ms)   delay(ms)\n";
    std::cout << "-------------------------------------------------\n";
    run(1);
    run(2);
    run(4);
    run(8);
    run(32);
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ex06b.cpp -o ex06b && ./ex06b

### 結果 ―― 容量1は、遅い

だいたい次のようになったはずです。

```
  capacity   time(ms)   idle(ms)   delay(ms)
-------------------------------------------------
       1        2356         546           36
       2        1994         179           45
       4        1815           5           85
       8        1820           5          151
      32        1828           5          175
```

**容量1では 2356ms。理想の 1800ms より 30% も遅い**です。
`idle` を見ると、受け取る側が 546ms も手待ちしていたことが分かります。

原因は、作る側が 105ms かかっている間です。
容量1では、作る側は**先に1個しか置けません**。受け取る側はその1個を 30ms で
使い切ってしまい、あとは手が空きます。

```
容量1（作る側が一度だけ遅れた）
作る側   P.P.P.P.PPPPPPPPPPP.P.P.P
行列     1 1 1 1 1..........1 1 1 1
受ける側 CCCCCCCCCCCC......CCCCCCCC     ← 手待ちが出る
                     ^^^^^^ ここが idle

容量4（同じでこぼこでも止まらない）
作る側   PPPP........PPPP........
行列     1234 4321   1234 4321
受ける側 CCCCCCCCCCCCCCCCCCCCCCCC     ← 途切れない
```

容量4では、速いうちに**貯金**ができています。作る側が遅れているあいだ、
受け取る側はその貯金を食べて働き続けられます。

> **容量とは、段どうしの「でこぼこ」を吸収するクッションである。**

### そして、大きくしても得はしない

容量4で `idle` は 5ms まで落ち、所要時間は理想の 1800ms に届いています。
**そこから先、8 にしても 32 にしても所要時間は変わりません。**

変わるのは遅れだけです。85ms → 151ms → 175ms と、悪くなる一方です。

```
        小さすぎる          ちょうどよい         大きすぎる
所要時間   遅い（手待ち）  →   理想に到達    →    理想のまま（変わらない）
遅れ       小さい          →   小さい        →    どんどん増える
メモリ     小さい          →   小さい        →    どんどん増える
```

つまり **「所要時間が理想に届く、いちばん小さい容量」が答え**です。
この例では 4 です。

## 6-3. 容量の決め方

ここまでを、実際に手を動かすときの手順にまとめます。

**① まず、一番遅い段を見つける**

スループットはそこで決まります。容量をいじっても、これは1ミリも変わりません
（6-1 で見たとおりです）。速くしたいなら、**まず一番遅い段を速くする**しかありません。

**② 容量は「でこぼこを吸収できる最小値」にする**

目安は次のとおりです。

```
容量 ≒ 前の段が一時的に遅れる最大の時間 ÷ 次の段が1個を処理する時間
```

6-2 の例なら `105ms ÷ 30ms ≒ 3.5` で、実測の 4 とだいたい合います。
とはいえ正確に計算する必要はありません。**1 から順に増やして、
所要時間が下がらなくなったところで止める**のが確実です。

**③ 「遅いから容量を増やす」は、たいてい間違い**

容量を増やして速くなるのは、**手待ちが出ているとき（`idle` が大きいとき）だけ**です。
それ以外の場合、増えるのは遅れとメモリだけです。

**④ 迷ったら、大きすぎるより小さすぎるほうがまし**

- 小さすぎる ⇒ 少し遅くなる。**測れば分かる**
- 大きすぎる ⇒ 遅れが増え、メモリが増え、しばらく走らせると落ちる。**測っても気づきにくい**

> **バックプレッシャは、問題を隠さずに表に出すしくみ。**
> 容量を無制限にするのは、問題を「メモリが尽きるまで先送りする」ことに等しい。

### キューの長さは、それ自体が情報

ここで1つ、本番で効いてくる話をしておきます。

**キューの長さを見れば、どこが詰まっているかが分かります。**

- あるキューが**いつも満杯** ⇒ その**すぐ下流**の段が追いついていない
- あるキューが**いつも空** ⇒ その**上流**が供給しきれていない

パイプラインのどこがボトルネックかを探すとき、
各段の時間を測るより先に、**キューの長さを1行表示してみる**ほうが早いことがよくあります。
これは発展課題4で実際に確かめます。

## 発展課題

1. 6-1 の設定（作る側が受け取る側より毎秒15個ぶん速い）で、容量を決めずに動かしたとします。
   1フレーム 384KB として、**メモリを 4GB 使い切るまで何分**でしょうか。
   また、その直前の時点で「遅れ」は何秒になっているでしょうか。

2. 3段（Read → Infer → Show）のパイプラインで、**真ん中の Infer だけが遅い**とします。
   キューが満杯になって Infer の前が詰まったとき、**Read 係はどうなるでしょうか。**
   バックプレッシャが「上流に伝わる」とはどういうことか、言葉で説明してください。

3. カメラの映像をその場で表示する場合、**遅れが大きいことは表示が止まるより悪い**ことがあります。
   このとき「満杯なら入れる側を待たせる」以外に、
   **「いちばん古いものを捨てて、新しいものを入れる」**という選択肢があります。
   - どんな用途なら許されるでしょうか
   - 逆に、**絶対に捨ててはいけない**のはどんな段でしょうか

4. 3段パイプラインで、キューが2本あります（Read→Infer と Infer→Show）。
   この**2本の長さを観察するだけ**で、どの段がボトルネックかを言い当てられるでしょうか。
   次の3つの場合について、2本のキューがそれぞれ長くなるか空になるか予測してください。
   - Infer が遅いとき
   - Show が遅いとき
   - Read が遅いとき

5. 「容量を増やしたら速くなった」という測定結果が出ることがあります。
   しかし**測り方によっては、速くなっていないのに速く見えてしまいます。**
   どういう測り方をすると、そうなってしまうでしょうか。
   （ヒント：最初の1個が出てくるまでの時間と、全部出終わるまでの時間）